In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define your persistent storage directory structure
DRIVE_BASE_DIR = '/content/drive/MyDrive/TGCN_cancer_driver/data/pan/string_850'
os.makedirs(DRIVE_BASE_DIR, exist_ok=True)

# Centralized data paths configuration
PATHS = {
    "raw_features": "/content/TGCN/data/pan/mut_features_miRNA_sub_du.txt",
    "ppi_network": "/content/TGCN/data/pan/string_850/STRING_850.txt",
    "filtered_ppi": "/content/TGCN/data/pan/string_850/mut_PPI_2.0.txt",

    # Persistent outputs routed directly to Google Drive
    "mut_data_pkl": os.path.join(DRIVE_BASE_DIR, 'mut_data_miRNA_sub_du_go_path_2.0.pkl'),
    "emb_ppi": os.path.join(DRIVE_BASE_DIR, 'mut_str_features_16_0.0001_2.0.pkl'),
    "emb_go": os.path.join(DRIVE_BASE_DIR, 'mut_str_features_16_0.0001_go_2.0.pkl'),
    "emb_path": os.path.join(DRIVE_BASE_DIR, 'mut_str_features_16_0.0001_path_2.0.pkl'),

    "data_67_ppi": os.path.join(DRIVE_BASE_DIR, 'data_67_0.0001_2.0.pkl'),
    "data_67_go": os.path.join(DRIVE_BASE_DIR, 'data_67_0.0001_go_2.0.pkl'),
    "data_67_path": os.path.join(DRIVE_BASE_DIR, 'data_67_0.0001_path_2.0.pkl'),

    "y_data": os.path.join(DRIVE_BASE_DIR, 'Y_796_data_2.0.pkl'),
    "k_sets": os.path.join(DRIVE_BASE_DIR, 'k_sets_796_2.0.pkl')
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!git clone https://github.com/wannaBMD/TGCN.git
%cd TGCN

fatal: destination path 'TGCN' already exists and is not an empty directory.
/content/TGCN


In [3]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
# 1. First, check your exact PyTorch and CUDA versions provided by Colab
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)

# 2. Install pre-compiled Geometric binaries directly mapped to the active Torch version
# We use --no-dependencies to prevent pip from getting stuck rebuilding existing packages
!pip install --no-cache-dir torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install --no-cache-dir torch-geometric==2.5.2

# 3. Install DGL explicitly without checking the global repository indexes
!pip install --no-cache-dir dgl -f https://data.dgl.ai/wheels/cu{torch.version.cuda.replace('.', '')}/repo.html

Torch version: 2.11.0+cpu
CUDA version: None
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cpu.html
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `pip install --no-cache-dir dgl -f https://data.dgl.ai/wheels/cu{torch.version.cuda.replace('.', '')}/repo.html'


In [4]:
!apt-get install unrar
!unrar x /content/TGCN/data/pan/string_850/go_2.0.rar /content/TGCN/data/pan/string_850/
!unrar x /content/TGCN/data/pan/string_850/path_2.0.rar /content/TGCN/data/pan/string_850/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/TGCN/data/pan/string_850/go_2.0.rar


Would you like to replace the existing file /content/TGCN/data/pan/string_850/go_2.0.tsv
460856203 bytes, modified on 2024-08-05 03:37
with a new one
460856203 bytes, modified on 2024-08-05 03:37

[Y]es, [N]o, [A]ll, n[E]ver, [R]ename, [Q]uit Y

Extracting  /content/TGCN/data/pan/string_850/go_2.0.tsv                 100%y
  OK 
All OK

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/TGCN/data/pan/string_850/path_2.0.rar


Would you like to replace the existing file /content/TGCN/data/pan/string_850/path_2.0.tsv
362407351 bytes, modified on 2024-08-05 03:42
with a new one
362

In [5]:
import os
import numpy as np
import pandas as pd

####Feature Extraction
It processes mutation frequencies, copy number variations, and epigenetic features to output a 51-dimensional matrix representing the pure biological signature of each gene.

In [6]:
import torch
gene_features = pd.read_table('/content/TGCN/data/pan/mut_features_miRNA_sub_du.txt', index_col=0, header =0, sep="\t")
#load PPI network
def load_network(file_path):
    ppi = pd.read_table(filepath_or_buffer=file_path, header=None, index_col=None, names=['source', 'target'], dtype='str', sep='\t')
    ppi_nodes = pd.concat([ppi['source'], ppi['target']], ignore_index=True)
    ppi_nodes = pd.DataFrame(ppi_nodes, columns=['nodes']).drop_duplicates()
    ppi_nodes.reset_index(drop=True, inplace=True)
    return ppi,ppi_nodes

#obtain genes that appear simultaneously in the gene set and PPI network
def get_intersection_genes(g_lst, ppi_nodes):
    p = ppi_nodes.iloc[:,0].values.tolist()
    p_nodes = list(set(p))
    print(len(p_nodes))
    g_intersection = list(set(g_lst) & set(p_nodes))
    return g_intersection

#retain the interactions where both source and target nodes are present in the gene list
def filter_ppi_with_intersect_nodes(nodes_lst,ppi_df):
    g_lst_df=pd.DataFrame(nodes_lst,columns=['g1'])
    m1=pd.merge(left=ppi_df,right=g_lst_df,left_on='source',right_on='g1',how='left')
    m1.dropna(how='any',inplace=True)
    m1.drop(['g1'],axis=1,inplace=True)

    m2=pd.merge(left=m1,right=g_lst_df,left_on='target',right_on='g1',how='left')
    m2.dropna(how='any',inplace=True)
    m2.drop(['g1'],axis=1,inplace=True)
    return m2

#obtain biological features of the filtered genes
def select_features(p_nodes):
    g_lst = list(p_nodes.iloc[:,0].values)
    print(len(g_lst))
    gene_list = list(gene_features.index.values)
    features = pd.DataFrame()
    for gene1 in gene_list:
        if gene1 in g_lst:
            features = pd.concat([features,gene_features.loc[gene1,:]], axis=1)
    features.index = [gene_features.columns]
    features.T.to_csv(path_or_buf=PATHS["raw_features"], sep='\t', header=True,index=True)

#filter gene and PPI
def data_filter_mut():
    ppi,ppi_nodes=load_network(PATHS["ppi_network"])
    print(ppi_nodes.shape)
    mut_g_lst = list(gene_features.index.values)
    print(len(mut_g_lst))
    g_intersection = get_intersection_genes(mut_g_lst, ppi_nodes)
    print(len(g_intersection))
    ppi = filter_ppi_with_intersect_nodes(g_intersection,ppi)
    ppi.to_csv(path_or_buf=PATHS["filtered_ppi"], sep='\t', header=False,index=False)
    p_nodes = pd.concat([ppi['source'], ppi['target']], ignore_index=True)
    p_nodes = pd.DataFrame(p_nodes, columns=['nodes']).drop_duplicates()
    p_nodes.reset_index(drop=True, inplace=True)
    select_features(p_nodes)
gene_features = pd.read_table(PATHS["raw_features"], index_col=0, header=0, sep="\t")
data_filter_mut()

(13472, 1)
10743
13472
10743
10743


####

In [7]:
import os
import numpy as np
import pandas as pd
import torch
from torch_geometric.nn import Node2Vec
from torch_geometric.data import Data
import networkx as nx
from torch_geometric.utils import from_networkx
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_featured_graph(network):

    G = nx.from_pandas_adjacency(network)

    G_adj = nx.convert_node_labels_to_integers(G, ordering='sorted', label_attribute='label')

    print(f'If the graph is connected graph: {nx.is_connected(G_adj)}')
    print(f'The number of connected components: {nx.number_connected_components(G_adj)}')

    graph = from_networkx(G_adj)
    assert graph.is_undirected() == True
    print(f'The edge index is {graph.edge_index}')

    return graph

def create_dataset():
    df = pd.read_table(PATHS["raw_features"], index_col=0, header=0, sep='\t')
    gene_index = list(df.index.values)
    features = torch.tensor(np.array(df)).to(torch.float32)

    ppi, ppi_nodes = load_network(PATHS["filtered_ppi"])
    sou_list = list(ppi['source'].values)
    tar_list = list(ppi['target'].values)

    # Optimized lookup using dictionaries
    gene_to_idx = {gene: idx for idx, gene in enumerate(gene_index)}
    sou_index = [gene_to_idx[sou] for sou in sou_list if sou in gene_to_idx]
    tar_index = [gene_to_idx[tar] for tar in tar_list if tar in gene_to_idx]

    edge_index = torch.tensor([sou_index, tar_index], dtype=torch.long)

    go_network = pd.read_csv('/content/TGCN/data/pan/string_850/go_2.0.tsv', sep='\t', index_col=0, header=0)
    go_graph = load_featured_graph(go_network)

    path_network = pd.read_csv('/content/TGCN/data/pan/string_850/path_2.0.tsv', sep='\t', index_col=0, header=0)
    path_graph = load_featured_graph(path_network)

    data = Data(x=features, edge_index=edge_index, go_index=go_graph.edge_index, path_index=path_graph.edge_index)
    return data

# Main execution gate check
required_files = [PATHS["mut_data_pkl"], PATHS["emb_ppi"], PATHS["emb_go"], PATHS["emb_path"]]
all_files_exist = all(os.path.exists(f) for f in required_files)

if all_files_exist:
    print("Found pre-processed Node2Vec embeddings in Google Drive! Loading directly to bypass training")
    data1 = torch.load(PATHS["mut_data_pkl"], map_location=device, weights_only=False)
else:
    print("Pre-processed arrays not found in Google Drive. Starting Node2Vec training runs")
    data1 = create_dataset()
    torch.save(data1, PATHS["mut_data_pkl"])

    def train(model, optimizer, loader):
        model.train()
        total_loss = 0
        for pos_rw, neg_rw in loader:
            optimizer.zero_grad()
            loss = model.loss(pos_rw.to(device), neg_rw.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        return total_loss / len(loader)

    # --- Setup Model 1 ---
    model1 = Node2Vec(data1.edge_index, embedding_dim=16, walk_length=80, context_size=5, walks_per_node=10, num_negative_samples=1, p=1, q=1, sparse=True).to(device)
    loader1 = model1.loader(batch_size=128, shuffle=True)
    optimizer1 = torch.optim.SparseAdam(list(model1.parameters()), lr=0.0001)
    for epoch in range(1, 1001): train(model1, optimizer1, loader1)
    model1.eval(); torch.save(model1(), PATHS["emb_ppi"])

    # --- Setup Model 2 ---
    model2 = Node2Vec(data1.go_index, embedding_dim=16, walk_length=80, context_size=5, walks_per_node=10, num_negative_samples=1, p=1, q=1, num_nodes=10743, sparse=True).to(device)
    loader2 = model2.loader(batch_size=128, shuffle=True)
    optimizer2 = torch.optim.SparseAdam(list(model2.parameters()), lr=0.0001)
    for epoch in range(1, 1001): train(model2, optimizer2, loader2)
    model2.eval(); torch.save(model2(), PATHS["emb_go"])

    # --- Setup Model 3 ---
    model3 = Node2Vec(data1.path_index, embedding_dim=16, walk_length=80, context_size=5, walks_per_node=10, num_negative_samples=1, p=1, q=1, num_nodes=10743, sparse=True).to(device)
    loader3 = model3.loader(batch_size=128, shuffle=True)
    optimizer3 = torch.optim.SparseAdam(list(model3.parameters()), lr=0.0001)
    for epoch in range(1, 1001): train(model3, optimizer3, loader3)
    model3.eval(); torch.save(model3(), PATHS["emb_path"])
    print("Node2Vec structural embeddings calculated and saved to Google Drive.")

Found pre-processed Node2Vec embeddings in Google Drive! Loading directly to bypass training


##Multivariate Feature Matrix COncatenation

In [8]:
if os.path.exists(PATHS["data_67_ppi"]) and os.path.exists(PATHS["data_67_go"]) and os.path.exists(PATHS["data_67_path"]):
    print("Matrix splits found in Google Drive. Skipping tensor concatenation step.")
else:
    gene_features = pd.read_table(PATHS["raw_features"], index_col=0, header=0, sep="\t")
    datax = torch.tensor(data=gene_features.values)

    # PPI Build
    datas = torch.load(PATHS["emb_ppi"]).cpu()
    data = torch.cat([datax, datas], 1)
    torch.save(data, PATHS["data_67_ppi"])

    # Semantic Similarity Build
    datap = torch.load(PATHS["emb_go"]).cpu()
    data2 = torch.cat([datax, datap], 1)
    torch.save(data2, PATHS["data_67_go"])

    # Pathway Build
    dataq = torch.load(PATHS["emb_path"]).cpu()
    data3 = torch.cat([datax, dataq], 1)
    torch.save(data3, PATHS["data_67_path"])
    print("67-dimensional matrices saved cleanly to Drive.")

Matrix splits found in Google Drive. Skipping tensor concatenation step.


In [9]:
import numpy as np
import pandas as pd
import pickle
import torch
import random
from torch_geometric.data import Data
cancerType = "pan"
#load labels of pan-cancer
if cancerType=="pan":
    df1 = pd.read_table('/content/TGCN/data/pan/string_850/mut_features_miRNA_sub_du_2.0.txt', index_col=0, header =0, sep="\t")
    gene_index = list(df1.index.values)
    df2 = pd.read_table('/content/TGCN/data/796true.txt', header=None, index_col=None, names=['name'], dtype='str')
    df3 = pd.read_table('/content/TGCN/data/2187false.txt', header=None, index_col=None, names=['name'], dtype='str')
#create label sets
driver_index = []
normal_index = []
unkown_index = []
for index,gene in enumerate(gene_index):
    flag = 0
    for driver in list(df2['name'].values):
        if gene==driver:
            driver_index.append(index)
            flag = 1
    for normal in list(df3['name'].values):
        if gene==normal:
            normal_index.append(index)
            flag = 1
    if flag==0:
        unkown_index.append(index)

def generate_bool_list(lst):
    return [bool(x) for x in lst]

def create_y_data():
    y_list = [0]*len(gene_index)
    for index,gene in enumerate(gene_index):
        for driver in list(df2['name'].values):
            if gene==driver:
                y_list[index] = 1
    y_te = generate_bool_list(y_list)
    y = torch.tensor(np.array(y_list).reshape(len(gene_index),-1))
    y_t = torch.tensor(np.array(y_te).reshape(len(gene_index),-1))
    y_data =Data(y=y,y_te=y_t)
    return y_data


y_data = create_y_data()
torch.save(y_data, PATHS["y_data"])
#divide dataset for ten five-fold cross-validations
def create_k_sets():
    data = pd.read_table("/content/TGCN/data/pan/string_850/mut_features_miRNA_sub_du_2.0.txt", index_col=0, header =0, sep="\t")
    length = len(data.values)
    index_array = np.arange(0,length,1)
    driver_chunk_size = len(driver_index) // 5
    normal_chunk_size = len(normal_index) // 5
    k_sets_list = []
    for i in range(10):
        #randomly shuffle array order
        random.shuffle(driver_index)
        random.shuffle(normal_index)
        #divide the dataset into five parts, the last part possibly containing additional elements
        k_sets_i = []
        for j in range(5):
            tr_list = [False]*length
            for suoyin1 in driver_index:
                tr_list[suoyin1] = True
            for suoyin2 in normal_index:
                tr_list[suoyin2] = True
            te_list = [False]*length
            driver_index1 = driver_chunk_size*j
            driver_index2 = driver_chunk_size*(j+1)
            normal_index1 = normal_chunk_size*j
            normal_index2 = normal_chunk_size*(j+1)
            if j<4:
                for r1 in range(driver_index1,driver_index2):
                    index3 = driver_index[r1]
                    te_list[index3] = True
                    tr_list[index3] = False
                for r2 in range(normal_index1,normal_index2):
                    index4 = normal_index[r2]
                    te_list[index4] = True
                    tr_list[index4] = False
            if j==4:
                for r1 in range(driver_index1,len(driver_index)):
                    index3 = driver_index[r1]
                    te_list[index3] = True
                    tr_list[index3] = False
                for r2 in range(normal_index1,len(normal_index)):
                    index4 = normal_index[r2]
                    te_list[index4] = True
                    tr_list[index4] = False

            inputs = []
            inputs.append(tr_list)
            inputs.append(te_list)
            k_sets_i.append(inputs)
        k_sets_list.append(k_sets_i)
    k_sets = np.array(k_sets_list)
    print(k_sets.shape)
    with open(PATHS["k_sets"], 'wb') as f:
        pickle.dump(k_sets, f)

create_k_sets()

if os.path.exists(PATHS["y_data"]) and os.path.exists(PATHS["k_sets"]):
    print("Label configurations and cross-validation masks found in Drive.")
else:
  y_data = create_y_data()
  torch.save(y_data, PATHS["y_data"])
  # torch.save(y_data, '../data/Specific cancer/string_850/Y_%s_2.0.pkl' %cancerType)
  #divide dataset for ten five-fold cross-validations
  def create_k_sets():
    data = pd.read_table("/content/TGCN/data/pan/string_850/mut_features_miRNA_sub_du_2.0.txt", index_col=0, header =0, sep="\t")
    length = len(data.values)
    index_array = np.arange(0,length,1)
    driver_chunk_size = len(driver_index) // 5
    normal_chunk_size = len(normal_index) // 5
    k_sets_list = []
    for i in range(10):
        #randomly shuffle array order
        random.shuffle(driver_index)
        random.shuffle(normal_index)
        #divide the dataset into five parts, the last part possibly containing additional elements
        k_sets_i = []
        for j in range(5):
            tr_list = [False]*length
            for suoyin1 in driver_index:
                tr_list[suoyin1] = True
            for suoyin2 in normal_index:
                tr_list[suoyin2] = True
            te_list = [False]*length
            driver_index1 = driver_chunk_size*j
            driver_index2 = driver_chunk_size*(j+1)
            normal_index1 = normal_chunk_size*j
            normal_index2 = normal_chunk_size*(j+1)
            if j<4:
                for r1 in range(driver_index1,driver_index2):
                    index3 = driver_index[r1]
                    te_list[index3] = True
                    tr_list[index3] = False
                for r2 in range(normal_index1,normal_index2):
                    index4 = normal_index[r2]
                    te_list[index4] = True
                    tr_list[index4] = False
            if j==4:
                for r1 in range(driver_index1,len(driver_index)):
                    index3 = driver_index[r1]
                    te_list[index3] = True
                    tr_list[index3] = False
                for r2 in range(normal_index1,len(normal_index)):
                    index4 = normal_index[r2]
                    te_list[index4] = True
                    tr_list[index4] = False

            inputs = []
            inputs.append(tr_list)
            inputs.append(te_list)
            k_sets_i.append(inputs)
        k_sets_list.append(k_sets_i)
    k_sets = np.array(k_sets_list)
    print(k_sets.shape)
    with open(PATHS["k_sets"], 'wb') as f:
        pickle.dump(k_sets, f)

    create_k_sets()
    print("Cross-validation sets successfully stored to Google Drive.")


(10, 5, 2, 10743)
Label configurations and cross-validation masks found in Drive.


##TGCN

In [ ]:
# ============================================================
# TGCN PREREQUISITE DATA-LOADING CELL
# Run this before the modified training code
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# Check that PATHS already exists
# ------------------------------------------------------------

if "PATHS" not in globals():
    raise NameError(
        "PATHS is not defined. "
        "Run your path-configuration cell first."
    )

# ------------------------------------------------------------
# Load three input views
# ------------------------------------------------------------

required_files = [
    PATHS["data_67_ppi"],
    PATHS["data_67_go"],
    PATHS["data_67_path"],
    PATHS["y_data"],
    PATHS["k_sets"],
    PATHS["mut_data_pkl"]
]

missing_files = [
    file_path
    for file_path in required_files
    if not os.path.exists(file_path)
]

if len(missing_files) > 0:
    print("The following required files are missing:")
    for file_path in missing_files:
        print(file_path)

    raise FileNotFoundError(
        "Create the missing preprocessing files first."
    )

data = torch.load(
    PATHS["data_67_ppi"],
    map_location=device,
    weights_only=False
).float().to(device)

data2 = torch.load(
    PATHS["data_67_go"],
    map_location=device,
    weights_only=False
).float().to(device)

data3 = torch.load(
    PATHS["data_67_path"],
    map_location=device,
    weights_only=False
).float().to(device)

# ------------------------------------------------------------
# Load labels
# ------------------------------------------------------------

Y_data = torch.load(
    PATHS["y_data"],
    map_location=device,
    weights_only=False
)

Y = torch.logical_or(
    Y_data.y.bool(),
    Y_data.y_te.bool()
).float().to(device)

# ------------------------------------------------------------
# Load cross-validation masks
# ------------------------------------------------------------

with open(PATHS["k_sets"], "rb") as handle:
    k_sets = pickle.load(handle)

# ------------------------------------------------------------
# Load PPI edge index
# ------------------------------------------------------------

e_data = torch.load(
    PATHS["mut_data_pkl"],
    map_location=device,
    weights_only=False
)

e_edge_index = e_data.edge_index.long().to(device)

# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

print("Prerequisite data loaded successfully.")
print("Device:", device)
print("data shape:", tuple(data.shape))
print("data2 shape:", tuple(data2.shape))
print("data3 shape:", tuple(data3.shape))
print("Y shape:", tuple(Y.shape))
print("PPI edge_index shape:", tuple(e_edge_index.shape))
print("k_sets shape:", np.asarray(k_sets).shape)

In [10]:
# ============================================================
# TGCN TRAINING, CROSS-VALIDATION, PREDICTIONS, AND ATTRIBUTION
# Replace your existing code from def train(mask): onward
# ============================================================

import os
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn import metrics
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    confusion_matrix
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

analysis_dir = (
    "/content/drive/MyDrive/"
    "TGCN_cancer_driver/TGCN_analysis"
)

os.makedirs(analysis_dir, exist_ok=True)

# ------------------------------------------------------------
# Ensure tensors are on the correct device
# ------------------------------------------------------------

data = data.to(device).float()
data2 = data2.to(device).float()
data3 = data3.to(device).float()
Y = Y.to(device).float()
e_edge_index = e_edge_index.to(device)

# ------------------------------------------------------------
# Load gene names
# ------------------------------------------------------------

feature_df = pd.read_table(
    PATHS["raw_features"],
    index_col=0,
    header=0,
    sep="\t"
)

gene_names = feature_df.index.astype(str).tolist()
N = len(gene_names)

assert data.shape[0] == N
assert data2.shape[0] == N
assert data3.shape[0] == N
assert Y.shape[0] == N

print("Number of genes:", N)
print("PPI-view shape:", tuple(data.shape))
print("GO-view shape:", tuple(data2.shape))
print("Pathway-view shape:", tuple(data3.shape))
print("Label shape:", tuple(Y.shape))

# ------------------------------------------------------------
# Training function
# ------------------------------------------------------------

def train(mask):
    model.train()
    optimizer.zero_grad()

    mask_t = torch.as_tensor(
        mask,
        dtype=torch.bool,
        device=device
    )

    prediction = model()

    loss = F.binary_cross_entropy_with_logits(
        prediction[mask_t],
        Y[mask_t]
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    return loss.item()

# ------------------------------------------------------------
# Testing function
# ------------------------------------------------------------

@torch.no_grad()
def test(mask):
    model.eval()

    mask_t = torch.as_tensor(
        mask,
        dtype=torch.bool,
        device=device
    )

    logits = model()

    probabilities = torch.sigmoid(
        logits[mask_t]
    ).flatten().cpu().numpy()

    labels = Y[mask_t].flatten().cpu().numpy().astype(int)

    auc_value = roc_auc_score(
        labels,
        probabilities
    )

    aupr_value = average_precision_score(
        labels,
        probabilities
    )

    return auc_value, aupr_value, probabilities, labels

# ------------------------------------------------------------
# Additional threshold-based metrics
# ------------------------------------------------------------

def calculate_binary_metrics(labels, probabilities):
    predicted_labels = (
        probabilities >= 0.5
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predicted_labels,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "accuracy": metrics.accuracy_score(
            labels,
            predicted_labels
        ),
        "precision": precision_score(
            labels,
            predicted_labels,
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            predicted_labels,
            zero_division=0
        ),
        "specificity": specificity,
        "f1": f1_score(
            labels,
            predicted_labels,
            zero_division=0
        ),
        "mcc": matthews_corrcoef(
            labels,
            predicted_labels
        ),
        "balanced_accuracy": balanced_accuracy_score(
            labels,
            predicted_labels
        )
    }

# ------------------------------------------------------------
# Cross-validation storage
# ------------------------------------------------------------

AUC = np.zeros((5, 5), dtype=np.float64)
AUPR = np.zeros((5, 5), dtype=np.float64)

oof_score_sum = np.zeros(N, dtype=np.float64)
oof_label_sum = np.zeros(N, dtype=np.float64)
oof_count = np.zeros(N, dtype=np.int32)

metric_records = []

# ------------------------------------------------------------
# Attribution storage
# ------------------------------------------------------------

gene_importance_sum = np.zeros(N, dtype=np.float64)
gene_importance_squared_sum = np.zeros(N, dtype=np.float64)
attribution_count = np.zeros(N, dtype=np.int32)

ppi_feature_importance_sum = np.zeros(
    data.shape[1],
    dtype=np.float64
)

go_feature_importance_sum = np.zeros(
    data2.shape[1],
    dtype=np.float64
)

path_feature_importance_sum = np.zeros(
    data3.shape[1],
    dtype=np.float64
)

number_of_attribution_folds = 0

# ------------------------------------------------------------
# Training and repeated five-fold cross-validation
# ------------------------------------------------------------

epochs = 550
time_start = time.time()

for repeat_index in range(5):

    for fold_index in range(5):

        tr_mask = np.asarray(
            k_sets[repeat_index][fold_index][0],
            dtype=bool
        )

        te_mask = np.asarray(
            k_sets[repeat_index][fold_index][1],
            dtype=bool
        )

        if np.logical_and(
            tr_mask,
            te_mask
        ).any():

            raise ValueError(
                "Training and test masks overlap."
            )

        if tr_mask.sum() == 0:
            raise ValueError(
                "Training mask is empty."
            )

        if te_mask.sum() == 0:
            raise ValueError(
                "Test mask is empty."
            )

        model = net(e_edge_index).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.0001,
            weight_decay=5e-4
        )

        training_losses = []

        for epoch in range(epochs):

            loss_value = train(tr_mask)
            training_losses.append(loss_value)

        auc_value, aupr_value, fold_pred, fold_labels = test(
            te_mask
        )

        AUC[repeat_index, fold_index] = auc_value
        AUPR[repeat_index, fold_index] = aupr_value

        fold_metrics = calculate_binary_metrics(
            fold_labels,
            fold_pred
        )

        fold_metrics.update({
            "repeat": repeat_index + 1,
            "fold": fold_index + 1,
            "roc_auc": auc_value,
            "aupr": aupr_value,
            "train_size": int(tr_mask.sum()),
            "test_size": int(te_mask.sum()),
            "final_train_loss": training_losses[-1]
        })

        metric_records.append(fold_metrics)

        print(
            f"Repeat {repeat_index + 1}, "
            f"fold {fold_index + 1}: "
            f"ROC-AUC = {auc_value:.4f}, "
            f"AUPR = {aupr_value:.4f}"
        )

        # --------------------------------------------------------
        # Store out-of-fold predictions
        # --------------------------------------------------------

        test_indices = np.where(te_mask)[0]

        if len(test_indices) != len(fold_pred):
            raise ValueError(
                "Test genes and predictions have different lengths."
            )

        oof_score_sum[test_indices] += fold_pred
        oof_label_sum[test_indices] += fold_labels
        oof_count[test_indices] += 1

        # --------------------------------------------------------
        # Gradient × input attribution
        # --------------------------------------------------------

        logits, g1, g2, g3 = model.get_outputs_and_grads(
            data,
            data2,
            data3,
            te_mask,
            Y
        )

        x01_cpu = data.detach().cpu()
        x02_cpu = data2.detach().cpu()
        x03_cpu = data3.detach().cpu()

        g1_cpu = g1.detach().cpu()
        g2_cpu = g2.detach().cpu()
        g3_cpu = g3.detach().cpu()

        grad_input_ppi = torch.abs(
            x01_cpu * g1_cpu
        ).numpy()

        grad_input_go = torch.abs(
            x02_cpu * g2_cpu
        ).numpy()

        grad_input_path = torch.abs(
            x03_cpu * g3_cpu
        ).numpy()

        gene_importance_ppi = (
            grad_input_ppi.sum(axis=1)
        )

        gene_importance_go = (
            grad_input_go.sum(axis=1)
        )

        gene_importance_path = (
            grad_input_path.sum(axis=1)
        )

        total_gene_importance = (
            gene_importance_ppi
            + gene_importance_go
            + gene_importance_path
        )

        gene_importance_sum[test_indices] += (
            total_gene_importance[test_indices]
        )

        gene_importance_squared_sum[test_indices] += (
            total_gene_importance[test_indices] ** 2
        )

        attribution_count[test_indices] += 1

        ppi_feature_importance_sum += (
            grad_input_ppi[test_indices].mean(axis=0)
        )

        go_feature_importance_sum += (
            grad_input_go[test_indices].mean(axis=0)
        )

        path_feature_importance_sum += (
            grad_input_path[test_indices].mean(axis=0)
        )

        number_of_attribution_folds += 1

        del model
        del optimizer

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ------------------------------------------------------------
# Aggregate out-of-fold predictions
# ------------------------------------------------------------

valid_prediction_genes = (
    oof_count > 0
)

oof_scores = np.full(
    N,
    np.nan,
    dtype=np.float64
)

oof_labels = np.full(
    N,
    np.nan,
    dtype=np.float64
)

oof_scores[valid_prediction_genes] = (
    oof_score_sum[valid_prediction_genes]
    / oof_count[valid_prediction_genes]
)

oof_labels[valid_prediction_genes] = (
    oof_label_sum[valid_prediction_genes]
    / oof_count[valid_prediction_genes]
)

# ------------------------------------------------------------
# Check label consistency
# ------------------------------------------------------------

rounded_labels = np.round(
    oof_labels[valid_prediction_genes]
)

inconsistent_labels = np.sum(
    np.abs(
        oof_labels[valid_prediction_genes]
        - rounded_labels
    ) > 1e-8
)

print(
    "Genes with inconsistent labels:",
    int(inconsistent_labels)
)

# ------------------------------------------------------------
# Save gene-level TGCN predictions
# ------------------------------------------------------------

prediction_table = pd.DataFrame({
    "gene": gene_names,
    "tgcn_score": oof_scores,
    "label": oof_labels,
    "prediction_count": oof_count
})

prediction_table["known_driver"] = (
    prediction_table["label"] == 1
)

prediction_table = prediction_table.sort_values(
    "tgcn_score",
    ascending=False,
    na_position="last"
)

prediction_file = os.path.join(
    analysis_dir,
    "tgcn_gene_predictions.csv"
)

prediction_table.to_csv(
    prediction_file,
    index=False
)

# ------------------------------------------------------------
# Save cross-validation metrics
# ------------------------------------------------------------

metrics_table = pd.DataFrame(
    metric_records
)

metrics_table.to_csv(
    os.path.join(
        analysis_dir,
        "cross_validation_metrics.csv"
    ),
    index=False
)

np.save(
    os.path.join(
        analysis_dir,
        "AUC_matrix.npy"
    ),
    AUC
)

np.save(
    os.path.join(
        analysis_dir,
        "AUPR_matrix.npy"
    ),
    AUPR
)

# ------------------------------------------------------------
# Aggregate gene-level attribution
# ------------------------------------------------------------

valid_attribution_genes = (
    attribution_count > 0
)

gene_importance_mean = np.full(
    N,
    np.nan,
    dtype=np.float64
)

gene_importance_std = np.full(
    N,
    np.nan,
    dtype=np.float64
)

gene_importance_mean[
    valid_attribution_genes
] = (
    gene_importance_sum[
        valid_attribution_genes
    ]
    / attribution_count[
        valid_attribution_genes
    ]
)

gene_importance_variance = (
    gene_importance_squared_sum[
        valid_attribution_genes
    ]
    / attribution_count[
        valid_attribution_genes
    ]
    - gene_importance_mean[
        valid_attribution_genes
    ] ** 2
)

gene_importance_variance = np.maximum(
    gene_importance_variance,
    0
)

gene_importance_std[
    valid_attribution_genes
] = np.sqrt(
    gene_importance_variance
)

attribution_table = pd.DataFrame({
    "gene": gene_names,
    "gene_importance_mean": gene_importance_mean,
    "gene_importance_std": gene_importance_std,
    "attribution_count": attribution_count
})

attribution_table = attribution_table.sort_values(
    "gene_importance_mean",
    ascending=False,
    na_position="last"
)

attribution_table.to_csv(
    os.path.join(
        analysis_dir,
        "gene_attribution_summary.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save average feature-level importance for each view
# ------------------------------------------------------------

if number_of_attribution_folds > 0:

    ppi_feature_mean = (
        ppi_feature_importance_sum
        / number_of_attribution_folds
    )

    go_feature_mean = (
        go_feature_importance_sum
        / number_of_attribution_folds
    )

    path_feature_mean = (
        path_feature_importance_sum
        / number_of_attribution_folds
    )

    maximum_features = max(
        len(ppi_feature_mean),
        len(go_feature_mean),
        len(path_feature_mean)
    )

    view_feature_table = pd.DataFrame({
        "feature_index": np.arange(
            maximum_features
        )
    })

    view_feature_table["ppi_importance"] = np.nan
    view_feature_table["go_importance"] = np.nan
    view_feature_table["path_importance"] = np.nan

    view_feature_table.loc[
        :len(ppi_feature_mean) - 1,
        "ppi_importance"
    ] = ppi_feature_mean

    view_feature_table.loc[
        :len(go_feature_mean) - 1,
        "go_importance"
    ] = go_feature_mean

    view_feature_table.loc[
        :len(path_feature_mean) - 1,
        "path_importance"
    ] = path_feature_mean

    view_feature_table.to_csv(
        os.path.join(
            analysis_dir,
            "view_feature_importance.csv"
        ),
        index=False
    )

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\nTraining completed.")
print("Elapsed time:", time.time() - time_start)
print("Mean ROC-AUC:", AUC.mean())
print("ROC-AUC standard deviation:", AUC.std())
print("Mean AUPR:", AUPR.mean())
print("AUPR standard deviation:", AUPR.std())
print(
    "Genes with predictions:",
    int(valid_prediction_genes.sum()),
    "/",
    N
)

print("\nTop 20 TGCN-ranked genes:")
print(
    prediction_table[
        [
            "gene",
            "tgcn_score",
            "label",
            "prediction_count"
        ]
    ].head(20)
)

print("\nSaved files:")
print(prediction_file)
print(
    os.path.join(
        analysis_dir,
        "cross_validation_metrics.csv"
    )
)
print(
    os.path.join(
        analysis_dir,
        "gene_attribution_summary.csv"
    )
)

NameError: name 'data' is not defined

##PPI network Analysis

In [ ]:
# ============================================================
# PPI NETWORK ANALYSIS OF TGCN-PRIORITIZED GENES
# ============================================================

import os
import numpy as np
import pandas as pd
import networkx as nx

from scipy.stats import mannwhitneyu
from scipy.stats import spearmanr

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

analysis_dir = (
    "/content/drive/MyDrive/"
    "TGCN_cancer_driver/TGCN_analysis"
)

prediction_file = os.path.join(
    analysis_dir,
    "tgcn_gene_predictions.csv"
)

ppi_file = PATHS["filtered_ppi"]

# ------------------------------------------------------------
# Load TGCN predictions
# ------------------------------------------------------------

prediction_table = pd.read_csv(
    prediction_file
)

prediction_table["gene"] = (
    prediction_table["gene"]
    .astype(str)
)

prediction_table = prediction_table.dropna(
    subset=["tgcn_score"]
)

prediction_table = prediction_table.sort_values(
    "tgcn_score",
    ascending=False
)

# Choose the primary list for analysis
TOP_K = 100

top_genes = set(
    prediction_table.head(TOP_K)["gene"]
)

# ------------------------------------------------------------
# Load the PPI network
# ------------------------------------------------------------

ppi = pd.read_csv(
    ppi_file,
    sep="\t",
    header=None,
    names=["source", "target"],
    dtype=str
)

ppi = ppi.dropna()

ppi["source"] = (
    ppi["source"]
    .astype(str)
    .str.strip()
)

ppi["target"] = (
    ppi["target"]
    .astype(str)
    .str.strip()
)

# Remove self-interactions and duplicate edges
ppi = ppi[
    ppi["source"] != ppi["target"]
]

ppi["edge_key"] = ppi.apply(
    lambda row: tuple(
        sorted(
            [row["source"], row["target"]]
        )
    ),
    axis=1
)

ppi = ppi.drop_duplicates(
    subset=["edge_key"]
)

ppi = ppi.drop(
    columns=["edge_key"]
)

# ------------------------------------------------------------
# Construct an undirected PPI graph
# ------------------------------------------------------------

G = nx.Graph()

G.add_edges_from(
    ppi[
        ["source", "target"]
    ].itertuples(
        index=False,
        name=None
    )
)

print("PPI network summary")
print("-------------------")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print(
    "Number of connected components:",
    nx.number_connected_components(G)
)

# ------------------------------------------------------------
# Calculate network properties
# ------------------------------------------------------------

degree = dict(G.degree())

betweenness = nx.betweenness_centrality(
    G,
    normalized=True
)

closeness = nx.closeness_centrality(G)

eigenvector = nx.eigenvector_centrality(
    G,
    max_iter=1000
)

pagerank = nx.pagerank(G)

clustering = nx.clustering(G)

core_number = nx.core_number(G)

# ------------------------------------------------------------
# Create network-statistics table
# ------------------------------------------------------------

network_table = pd.DataFrame({
    "gene": list(G.nodes()),
    "degree": [
        degree[gene]
        for gene in G.nodes()
    ],
    "betweenness": [
        betweenness[gene]
        for gene in G.nodes()
    ],
    "closeness": [
        closeness[gene]
        for gene in G.nodes()
    ],
    "eigenvector": [
        eigenvector[gene]
        for gene in G.nodes()
    ],
    "pagerank": [
        pagerank[gene]
        for gene in G.nodes()
    ],
    "clustering": [
        clustering[gene]
        for gene in G.nodes()
    ],
    "core_number": [
        core_number[gene]
        for gene in G.nodes()
    ]
})

# Add TGCN scores and labels
network_table = network_table.merge(
    prediction_table[
        [
            "gene",
            "tgcn_score",
            "label",
            "known_driver",
            "prediction_count"
        ]
    ],
    on="gene",
    how="left"
)

network_table["is_top100"] = (
    network_table["gene"].isin(top_genes)
)

network_table["group"] = "other_network_gene"

network_table.loc[
    network_table["known_driver"] == True,
    "group"
] = "known_driver"

network_table.loc[
    network_table["is_top100"] == True,
    "group"
] = "top100_TGCN"

# ------------------------------------------------------------
# Save complete network table
# ------------------------------------------------------------

network_table.to_csv(
    os.path.join(
        analysis_dir,
        "ppi_network_statistics.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Group-level summary
# ------------------------------------------------------------

group_summary = network_table.groupby(
    "group"
)[
    [
        "degree",
        "betweenness",
        "closeness",
        "eigenvector",
        "pagerank",
        "clustering",
        "core_number",
        "tgcn_score"
    ]
].agg(
    [
        "count",
        "mean",
        "median",
        "std"
    ]
)

group_summary.to_csv(
    os.path.join(
        analysis_dir,
        "ppi_group_summary.csv"
    )
)

print("\nGroup-level network summary")
print(group_summary)

# ------------------------------------------------------------
# Compare top-100 genes with all other scored network genes
# ------------------------------------------------------------

scored_network_table = network_table[
    network_table["tgcn_score"].notna()
].copy()

top_group = scored_network_table[
    scored_network_table["is_top100"] == True
]

other_group = scored_network_table[
    scored_network_table["is_top100"] == False
]

network_properties = [
    "degree",
    "betweenness",
    "closeness",
    "eigenvector",
    "pagerank",
    "clustering",
    "core_number"
]

comparison_records = []

for property_name in network_properties:

    top_values = top_group[
        property_name
    ].dropna()

    other_values = other_group[
        property_name
    ].dropna()

    statistic, p_value = mannwhitneyu(
        top_values,
        other_values,
        alternative="two-sided"
    )

    comparison_records.append({
        "property": property_name,
        "top100_median": top_values.median(),
        "other_median": other_values.median(),
        "top100_mean": top_values.mean(),
        "other_mean": other_values.mean(),
        "mann_whitney_u": statistic,
        "p_value": p_value
    })

comparison_table = pd.DataFrame(
    comparison_records
)

comparison_table.to_csv(
    os.path.join(
        analysis_dir,
        "top100_vs_other_network_comparison.csv"
    ),
    index=False
)

print("\nTop-100 versus other network genes")
print(comparison_table)

# ------------------------------------------------------------
# Correlation between TGCN score and network properties
# ------------------------------------------------------------

correlation_records = []

for property_name in network_properties:

    valid_rows = scored_network_table[
        [
            "tgcn_score",
            property_name
        ]
    ].dropna()

    rho, p_value = spearmanr(
        valid_rows["tgcn_score"],
        valid_rows[property_name]
    )

    correlation_records.append({
        "property": property_name,
        "spearman_rho": rho,
        "p_value": p_value,
        "number_of_genes": len(valid_rows)
    })

correlation_table = pd.DataFrame(
    correlation_records
)

correlation_table.to_csv(
    os.path.join(
        analysis_dir,
        "tgcn_score_network_correlations.csv"
    ),
    index=False
)

print("\nCorrelation between TGCN score and network properties")
print(correlation_table)

# ------------------------------------------------------------
# Rank genes by each network property
# ------------------------------------------------------------

for property_name in network_properties:

    output_name = (
        "top_genes_by_"
        + property_name
        + ".csv"
    )

    ranked_genes = network_table.sort_values(
        property_name,
        ascending=False
    )[
        [
            "gene",
            property_name,
            "tgcn_score",
            "known_driver",
            "is_top100"
        ]
    ].head(100)

    ranked_genes.to_csv(
        os.path.join(
            analysis_dir,
            output_name
        ),
        index=False
    )

# ------------------------------------------------------------
# Save the PPI subnetwork containing top-100 genes
# ------------------------------------------------------------

top_subgraph_nodes = [
    gene
    for gene in top_genes
    if gene in G.nodes()
]

top_subgraph = G.subgraph(
    top_subgraph_nodes
).copy()

top_subgraph_edges = pd.DataFrame(
    list(
        top_subgraph.edges()
    ),
    columns=["source", "target"]
)

top_subgraph_edges.to_csv(
    os.path.join(
        analysis_dir,
        "top100_TGCN_PPI_edges.csv"
    ),
    index=False
)

top_subgraph_nodes_table = network_table[
    network_table["gene"].isin(
        top_subgraph_nodes
    )
].copy()

top_subgraph_nodes_table.to_csv(
    os.path.join(
        analysis_dir,
        "top100_TGCN_PPI_nodes.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Print the most important results
# ------------------------------------------------------------

print("\nTop TGCN genes with PPI statistics")
print(
    network_table[
        network_table["is_top100"] == True
    ].sort_values(
        "tgcn_score",
        ascending=False
    )[
        [
            "gene",
            "tgcn_score",
            "degree",
            "betweenness",
            "closeness",
            "pagerank",
            "core_number",
            "known_driver"
        ]
    ].head(20)
)

print("\nPPI analysis completed.")
print(
    "Results saved in:",
    analysis_dir
)

#To obtain Global Feature Importance

In [ ]:
def compute_global_feature_importance(model, x01, x02, x03, mask, Y):
    model.eval()

    x01 = x01.clone().detach().requires_grad_(True)
    x02 = x02.clone().detach().requires_grad_(True)
    x03 = x03.clone().detach().requires_grad_(True)

    edge_index = model.e_edge_index

    x1 = model.T1(x01)
    x2 = torch.relu(x1 + model.lin11(x01))
    x4 = model.T2(x2)

    x6 = model.T1(x02)
    x7 = torch.relu(x6 + model.lin21(x02))
    x9 = model.T2(x7)

    x11 = model.T1(x03)
    x12 = torch.relu(x11 + model.lin31(x03))
    x14 = model.T2(x12)

    x19 = x4 + x9 + x14
    logits = model.GNN(x19, edge_index)

    mask_t = torch.tensor(mask, dtype=torch.bool, device=logits.device)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits[mask_t], Y[mask_t])

    model.zero_grad()
    loss.backward()

    grad_x01 = x01.grad.detach()
    grad_x02 = x02.grad.detach()
    grad_x03 = x03.grad.detach()

    imp_ppi = (x01.detach() * grad_x01).abs().mean(dim=0).cpu().numpy()
    imp_go = (x02.detach() * grad_x02).abs().mean(dim=0).cpu().numpy()
    imp_path = (x03.detach() * grad_x03).abs().mean(dim=0).cpu().numpy()

    return imp_ppi, imp_go, imp_path

In [ ]:
global_imp_ppi_all = []
global_imp_go_all = []
global_imp_path_all = []

for i in range(5):
    for cv_run in range(5):
        tr_mask = k_sets[i][cv_run][0]
        te_mask = k_sets[i][cv_run][1]

        model = net(e_edge_index).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=5e-4)

        for t in range(epochs):
            train(tr_mask)

        AUC[i][cv_run], AUPR[i][cv_run] = test(te_mask)

        imp_ppi, imp_go, imp_path = compute_global_feature_importance(
            model, data, data2, data3, te_mask, Y
        )

        global_imp_ppi_all.append(imp_ppi)
        global_imp_go_all.append(imp_go)
        global_imp_path_all.append(imp_path)

global_imp_ppi = np.mean(np.stack(global_imp_ppi_all, axis=0), axis=0)
global_imp_go = np.mean(np.stack(global_imp_go_all, axis=0), axis=0)
global_imp_path = np.mean(np.stack(global_imp_path_all, axis=0), axis=0)

ignore below analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_PATH = "/content/drive/MyDrive/TGCN_cancer_driver/TGCN_analysis"

# Load gene index
df1 = pd.read_table(
    '/content/TGCN/data/pan/string_850/mut_features_miRNA_sub_du_2.0.txt',
    index_col=0,
    header=0,
    sep="\t"
)
gene_index = list(df1.index.values)
n_genes = len(gene_index)
gene_to_idx = {g: i for i, g in enumerate(gene_index)}

# Load driver gene list
df2 = pd.read_table(
    '/content/TGCN/data/796true.txt',
    header=None,
    index_col=None,
    names=['name'],
    dtype='str'
)
driver_genes = set(df2['name'].values)

n_folds = 5
n_cv = 5

# Store gene-level mutation importance per gene (accumulate across folds)
gene_mut_importance_accum = np.zeros(n_genes)
gene_mut_count_accum = np.zeros(n_genes)

# For feature-level importance, just collect all and average
feat_ppi_list = []
feat_go_list = []
feat_path_list = []

for i in range(n_folds):
    for cv_run in range(n_cv):
        # Load feature importance per view
        ppi_path = os.path.join(BASE_PATH, f"feat_import_ppi_fold{i}_cv{cv_run}.npy")
        go_path  = os.path.join(BASE_PATH, f"feat_import_go_fold{i}_cv{cv_run}.npy")
        path_path= os.path.join(BASE_PATH, f"feat_import_path_fold{i}_cv{cv_run}.npy")

        feat_ppi = np.load(ppi_path)  # should be (67,)
        feat_go  = np.load(go_path)
        feat_path= np.load(path_path)

        # Check shapes
        print(f"Fold {i}, CV {cv_run}:")
        print("  feat_ppi shape:", feat_ppi.shape)
        print("  feat_go shape:", feat_go.shape)
        print("  feat_path shape:", feat_path.shape)

        feat_ppi_list.append(feat_ppi)
        feat_go_list.append(feat_go)
        feat_path_list.append(feat_path)

        # Load gene importance (test genes only)
        gene_imp_path = os.path.join(BASE_PATH, f"gene_importance_fold{i}_cv{cv_run}.npy")
        gene_imp = np.load(gene_imp_path)  # (num_test_genes,)

        # We need to know which genes are in this test set.
        # If you didn't save the test gene indices, you can reconstruct from te_mask.
        # For now, assume you re-load te_mask from k_sets later.
        # We'll skip gene-level accumulation here and do it below with k_sets.

# Average feature importance across folds
feat_importance_ppi = np.mean(np.stack(feat_ppi_list), axis=0)
feat_importance_go  = np.mean(np.stack(feat_go_list), axis=0)
feat_importance_path= np.mean(np.stack(feat_path_list), axis=0)

print("Final feature importance shapes:")
print("  PPI:", feat_importance_ppi.shape)
print("  GO:", feat_importance_go.shape)
print("  PATH:", feat_importance_path.shape)

##Analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_PATH = "/content/drive/MyDrive/TGCN_cancer_driver/TGCN_analysis"
DATA_PATH = "/content/drive/MyDrive/TGCN_cancer_driver/TGCN"

# Load gene index and driver list
df1 = pd.read_table(
    '/content/TGCN/data/pan/string_850/mut_features_miRNA_sub_du_2.0.txt',
    index_col=0, header=0, sep="\t"
)
gene_index = list(df1.index.values)
n_genes = len(gene_index)
gene_to_idx = {g: i for i, g in enumerate(gene_index)}

df2 = pd.read_table(
    '/content/TGCN/data/796true.txt',
    header=None, index_col=None, names=['name'], dtype='str'
)
driver_genes = set(df2['name'].values)

# Load full data (PPI view) for gene-level mutation importance
data_full = torch.load(os.path.join(DATA_PATH, "data_67_ppi.pth"), weights_only=False)
data_full = data_full.cpu().numpy()  # (n_genes, 67)

n_folds = 5
n_cv = 5

feat_ppi_list = []
feat_go_list = []
feat_path_list = []

# For gene-level mutation importance accumulation
gene_mut_importance_accum = np.zeros(n_genes)
gene_mut_count_accum = np.zeros(n_genes)

for i in range(n_folds):
    for cv_run in range(n_cv):
        # Load feature importance per view
        ppi_path = os.path.join(BASE_PATH, f"feat_import_ppi_fold{i}_cv{cv_run}.npy")
        go_path  = os.path.join(BASE_PATH, f"feat_import_go_fold{i}_cv{cv_run}.npy")
        path_path= os.path.join(BASE_PATH, f"feat_import_path_fold{i}_cv{cv_run}.npy")

        feat_ppi = np.load(ppi_path)
        feat_go  = np.load(go_path)
        feat_path= np.load(path_path)

        feat_ppi_list.append(feat_ppi)
        feat_go_list.append(feat_go)
        feat_path_list.append(feat_path)

        # Load gene importance (test genes only)
        gene_imp_path = os.path.join(BASE_PATH, f"gene_importance_fold{i}_cv{cv_run}.npy")
        gene_imp = np.load(gene_imp_path)  # (num_test_genes,)

        # We need te_mask to map these back to full gene indices.
        # If you didn't save te_mask, you can reconstruct from k_sets:
        import pickle
        with open('/content/TGCN/data/pan/k_sets.pkl', 'rb') as f:
            k_sets = pickle.load(f)

        te_mask = np.array(k_sets[i][cv_run][1])  # boolean array of length n_genes
        test_gene_indices = np.where(te_mask)[0]  # indices in full gene_index

        # Accumulate gene-level mutation importance (more detailed version below)
        # For now, just accumulate total gene importance
        gene_mut_importance_accum[test_gene_indices] += gene_imp
        gene_mut_count_accum[test_gene_indices] += 1

# Average feature importance across folds
feat_importance_ppi = np.mean(np.stack(feat_ppi_list), axis=0)
feat_importance_go  = np.mean(np.stack(feat_go_list), axis=0)
feat_importance_path= np.mean(np.stack(feat_path_list), axis=0)

print("Feature importance shapes:")
print("  PPI:", feat_importance_ppi.shape)
print("  GO:", feat_importance_go.shape)
print("  PATH:", feat_importance_path.shape)

# Average gene importance across folds
gene_mut_importance_avg = gene_mut_importance_accum / np.maximum(gene_mut_count_accum, 1)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Adjust this path to your actual Drive path
BASE_PATH = "/content/drive/MyDrive/TGCN_cancer_driver/TGCN_analysis"

# Load gene index (same as in your training code)
df1 = pd.read_table(
    '/content/TGCN/data/pan/string_850/mut_features_miRNA_sub_du_2.0.txt',
    index_col=0,
    header=0,
    sep="\t"
)
gene_index = list(df1.index.values)
n_genes = len(gene_index)
gene_to_idx = {g: i for i, g in enumerate(gene_index)}

# Load driver gene list (same as in your training code)
df2 = pd.read_table(
    '/content/TGCN/data/796true.txt',
    header=None,
    index_col=None,
    names=['name'],
    dtype='str'
)
driver_genes = set(df2['name'].values)

# Number of folds/cVs you trained
n_folds = 5
n_cv = 5

# Containers to aggregate across folds
all_gene_importance = []
all_te_masks = []
all_grad_ppi = []
all_grad_go = []
all_grad_path = []

for i in range(n_folds):
    for cv_run in range(n_cv):
        # Load gene importance
        gene_imp_path = os.path.join(
            BASE_PATH, f"gene_importance_fold{i}_cv{cv_run}.npy"
        )
        gene_imp = np.load(gene_imp_path)
        all_gene_importance.append(gene_imp)

        # Load feature importance per view (mean abs gradient per feature)
        ppi_path = os.path.join(BASE_PATH, f"feat_import_ppi_fold{i}_cv{cv_run}.npy")
        go_path  = os.path.join(BASE_PATH, f"feat_import_go_fold{i}_cv{cv_run}.npy")
        path_path= os.path.join(BASE_PATH, f"feat_import_path_fold{i}_cv{cv_run}.npy")

        feat_ppi = np.load(ppi_path)  # (67,)
        feat_go  = np.load(go_path)
        feat_path= np.load(path_path)

        all_grad_ppi.append(feat_ppi)
        all_grad_go.append(feat_go)
        all_grad_path.append(feat_path)

# Aggregate across folds: mean importance
all_gene_importance = np.vstack(all_gene_importance)  # (n_folds*n_cv, n_genes_test_in_fold)
# But gene_importance is only for test genes in each fold; we'll align at gene level later.

# For feature-level analysis, just average across folds
feat_importance_ppi = np.mean(np.stack(all_grad_ppi), axis=0)  # (67,)
feat_importance_go  = np.mean(np.stack(all_grad_go), axis=0)
feat_importance_path= np.mean(np.stack(all_grad_path), axis=0)

In [ ]:


n_cancers = 16
# Features per cancer: mutation, methylation, expression
features_per_cancer = 3

# Initialize importance per cancer type
importance_per_cancer = np.zeros(n_cancers)

for t in range(n_cancers):
    base_idx = t * features_per_cancer  # 0, 3, 6, ...
    indices = [base_idx, base_idx+1, base_idx+2]  # mutation, methylation, expression for cancer t

    # Mean absolute gradient for these 3 features, across all test genes
    importance_per_cancer[t] = np.mean(
        np.abs(grad_input_1_te[:, indices])
    )